# DATA 266 Lab 2 - Part 2: Fine-tuning Stable Diffusion 1.5 (Ritika Neema)

Pair_Programming_Team_05. Two runs on the same data, steps and seed: **LoRA rank 8** and **DoRA rank 8**
(weight-decomposed LoRA), both on the UNet attention projections of Stable Diffusion 1.5, trained on
`reach-vb/pokemon-blip-captions`. Sneha's runs compare LoRA ranks 4 and 16; mine compare LoRA with
DoRA at a fixed rank. Every number below is printed by this notebook.

Runs on one 16 GB GPU (Colab or Kaggle T4). Secrets: `HF_TOKEN` (write) and, on Colab, `GH_TOKEN`
to clone the private repo so the committed prompt file is read and the commit is printed.

In [2]:
# Cell 1: environment, repo checkout, versions, GPU and seed.
import importlib, importlib.metadata, os, subprocess, sys

# Colab ships an old torchao that recent peft refuses to run with; nothing here uses torchao, so remove it.
try:
    if tuple(int(x) for x in importlib.metadata.version("torchao").split(".")[:2]) < (0, 16):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
        if "peft" in sys.modules:
            raise RuntimeError("torchao was removed; restart the session (Runtime > Restart session) and run again")
except importlib.metadata.PackageNotFoundError:
    pass

for _pkg, _mod in [("diffusers", "diffusers"), ("peft", "peft"), ("torchmetrics", "torchmetrics"),
                   ("torch-fidelity", "torch_fidelity"), ("lpips", "lpips"), ("datasets", "datasets")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

REPO = "snehas-SJSU/data266-lab2-team05"
if not os.path.exists("eval/p2_prompts.json"):
    for up in ("..", "../.."):
        if os.path.exists(os.path.join(up, "eval/p2_prompts.json")):
            os.chdir(up); break
    else:
        if not os.path.exists("data266-lab2-team05"):
            token = get_secret("GH_TOKEN")
            if not token:
                raise RuntimeError(
                    "Cannot find eval/p2_prompts.json. Either (a) add a Colab secret named GH_TOKEN (a GitHub token with "
                    "read access to {REPO}) and switch on 'Notebook access' for it, or (b) upload the repo's eval/ "
                    "folder to /content/eval/ and re-run this cell.")
            subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/{REPO}.git"], check=True)
        os.chdir("data266-lab2-team05")

import json, time, random, hashlib, platform
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import diffusers, peft, transformers, torchmetrics
from PIL import Image

SEED = 266            # my seed (SJSU ID): training batches/noise, pair sampling
SPLIT_SEED = 266      # the lab fixes the 90/10 split to seed 266
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit      :", commit or "n/a")
print("python      :", platform.python_version(), "| torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| peft", peft.__version__, "| transformers", transformers.__version__, "| torchmetrics", torchmetrics.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("seed        :", SEED, "| split seed:", SPLIT_SEED)
DEV = "cuda"

commit      : 76f29f5
python      : 3.13.16 | torch 2.11.0+cu130 | diffusers 0.42.0 | peft 0.21.2 | transformers 5.20.2 | torchmetrics 1.9.0
GPU         : Tesla T4
seed        : 266 | split seed: 266


In [3]:
# Cell 2: configuration. Both runs share every setting except the adapter type.
BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DATASET = "reach-vb/pokemon-blip-captions"          # license: CC BY-NC-SA 4.0 (dataset card)
RES, STEPS, BATCH, GRAD_ACCUM, LR = 512, 1200, 2, 2, 1e-4   # effective batch 4
TARGETS = ["to_q", "to_k", "to_v", "to_out.0"]
RUNS = {"lora_r8": dict(rank=8, use_dora=False), "dora_r8": dict(rank=8, use_dora=True)}
RUN_NOW = [r for r in os.environ.get("P2_RUNS", ",".join(RUNS)).split(",") if r]   # e.g. one run per session
CKPT_EVERY = 400
GEN_STEPS, GUIDANCE = 30, 7.5
OUT = os.environ.get("P2_OUT", "outputs/p2_ritika"); os.makedirs(OUT, exist_ok=True)
HF_TOKEN = get_secret("HF_TOKEN")
from huggingface_hub import HfApi
API = HfApi(token=HF_TOKEN) if HF_TOKEN else None
HF_USER = API.whoami()["name"] if API else None
print("runs this session:", RUN_NOW, "| HF user:", HF_USER, "| out:", OUT)
PROMPTS = json.load(open("eval/p2_prompts.json"))
print(pd.DataFrame(PROMPTS)["type"].value_counts().to_dict())

runs this session: ['lora_r8', 'dora_r8'] | HF user: neema30 | out: outputs/p2_ritika
{'in-domain': 10, 'out-of-domain': 5, 'compositional': 5}


In [4]:
# Cell 3: data preparation. Drop corrupted and duplicate images, split 90/10 with SPLIT_SEED (266), resize to 512.
import io
from datasets import load_dataset, Image as HFImage

raw = load_dataset(DATASET, split="train").cast_column("image", HFImage(decode=False))
kept, corrupted, exact_dup, near_dup = [], 0, 0, 0
seen_exact, seen_near = set(), set()
for ex in raw:
    try:
        im = Image.open(io.BytesIO(ex["image"]["bytes"])); im.load(); im = im.convert("RGB")
        if min(im.size) < 64 or not ex["text"].strip():
            raise ValueError("too small or no caption")
    except Exception:
        corrupted += 1; continue
    h = hashlib.md5(im.tobytes()).hexdigest()
    small = np.asarray(im.convert("L").resize((16, 16), Image.BILINEAR), dtype=np.float32)
    a = "".join("1" if v else "0" for v in (small > small.mean()).flatten())   # 256-bit average hash
    if h in seen_exact:
        exact_dup += 1; continue
    if a in seen_near:
        near_dup += 1; continue
    seen_exact.add(h); seen_near.add(a)
    kept.append((im, ex["text"].strip()))

def to512(im):
    w, h = im.size; s = RES / min(w, h)
    im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
    l, t = (im.width - RES) // 2, (im.height - RES) // 2
    return im.crop((l, t, l + RES, t + RES))

perm = np.random.default_rng(SPLIT_SEED).permutation(len(kept))
n_test = round(0.1 * len(kept))
TEST = [(to512(kept[i][0]), kept[i][1]) for i in perm[:n_test]]
TRAIN = [(to512(kept[i][0]), kept[i][1]) for i in perm[n_test:]]
os.makedirs(f"{OUT}/test_images", exist_ok=True)
for k, (im, _) in enumerate(TEST):
    im.save(f"{OUT}/test_images/test_{k:03d}.png")
print(f"raw {len(raw)} | corrupted removed {corrupted} | exact duplicates removed {exact_dup} | "
      f"near duplicates removed {near_dup} | kept {len(kept)} -> train {len(TRAIN)}, test {len(TEST)}")

README.md:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

data/train-00000-of-00001-566cc9b19d7203(…): reconstructing file:   0%|          |  0.00B / 99.7MB            

data/train-00000-of-00001-566cc9b19d7203(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/833 [00:00<?, ? examples/s]

raw 833 | corrupted removed 0 | exact duplicates removed 0 | near duplicates removed 2 | kept 831 -> train 748, test 83


In [5]:
# Cell 4: cache VAE latents and text embeddings once, so both runs see identical inputs and only the UNet trains.
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer

tok = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
te = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=torch.float16).to(DEV)
vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float16).to(DEV)
lat, emb = [], []
with torch.no_grad():
    for i in range(0, len(TRAIN), 16):
        ims, caps = zip(*TRAIN[i:i + 16])
        x = torch.stack([torch.from_numpy(np.asarray(im, dtype=np.float32) / 127.5 - 1).permute(2, 0, 1) for im in ims])
        lat.append(vae.encode(x.to(DEV, torch.float16)).latent_dist.sample() * vae.config.scaling_factor)
        ids = tok(list(caps), padding="max_length", max_length=tok.model_max_length, truncation=True,
                  return_tensors="pt").input_ids.to(DEV)
        emb.append(te(ids)[0])
LATENTS, TEXT_EMB = torch.cat(lat), torch.cat(emb)
del te, vae; torch.cuda.empty_cache()
print("latents", tuple(LATENTS.shape), "| text embeddings", tuple(TEXT_EMB.shape))

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

text_encoder/model.safetensors: reconstructing file:   0%|          |  0.00B /  492MB            

... (3 outputs left out)

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

latents (748, 4, 64, 64) | text embeddings (748, 77, 768)


In [6]:
# Cell 5: training. Same sampled batches, noise and timesteps for both runs (generators seeded with SEED).
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import save_file, load_file

def lora_config(cfg):
    return LoraConfig(r=cfg["rank"], lora_alpha=cfg["rank"], init_lora_weights="gaussian",
                      target_modules=TARGETS, use_dora=cfg["use_dora"])

def save_adapter(unet, cfg, path, stats=None):
    os.makedirs(path, exist_ok=True)
    save_file({k: v.detach().half().contiguous() for k, v in get_peft_model_state_dict(unet).items()},
              f"{path}/adapter_model.safetensors")
    json.dump(dict(rank=cfg["rank"], use_dora=cfg["use_dora"], target_modules=TARGETS, base_model=BASE,
                   **(stats or {})), open(f"{path}/adapter_config.json", "w"), indent=1)

def push(path, repo, msg):
    if API:
        API.create_repo(repo, exist_ok=True, private=False)
        API.upload_folder(folder_path=path, repo_id=repo, commit_message=msg)

def train(name, cfg):
    unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16).to(DEV)
    unet.requires_grad_(False)
    unet.enable_gradient_checkpointing()
    unet.add_adapter(lora_config(cfg))
    params = [p for p in unet.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()                      # trainable weights in fp32, frozen UNet in fp16
    n_params = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)
    scaler = torch.amp.GradScaler("cuda")
    sched = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
    g_idx, g_gpu = torch.Generator().manual_seed(SEED), torch.Generator(device=DEV).manual_seed(SEED)
    torch.cuda.reset_peak_memory_stats(); t0 = time.time(); losses = []
    for step in range(1, STEPS + 1):
        for _ in range(GRAD_ACCUM):
            idx = torch.randint(0, len(LATENTS), (BATCH,), generator=g_idx).to(DEV)
            x0 = LATENTS[idx]
            noise = torch.randn(x0.shape, generator=g_gpu, device=DEV, dtype=x0.dtype)
            t = torch.randint(0, sched.config.num_train_timesteps, (BATCH,), generator=g_gpu, device=DEV)
            with torch.autocast("cuda", dtype=torch.float16):
                pred = unet(sched.add_noise(x0, noise, t), t, TEXT_EMB[idx]).sample
            loss = F.mse_loss(pred.float(), noise.float()) / GRAD_ACCUM
            scaler.scale(loss).backward()
            losses.append(loss.item() * GRAD_ACCUM)
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if step % 100 == 0:
            print(f"{name} step {step}/{STEPS} loss(avg last 100) {np.mean(losses[-100 * GRAD_ACCUM:]):.4f} "
                  f"{(time.time() - t0) / 60:.1f} min")
        if step % CKPT_EVERY == 0 and step < STEPS and HF_USER:
            save_adapter(unet, cfg, f"{OUT}/{name}")
            push(f"{OUT}/{name}", f"{HF_USER}/data266-lab2-p2-ritika-{name.replace('_', '-')}", f"checkpoint step {step}")
    stats = dict(trainable_params=n_params, lr=LR, steps=STEPS, batch=BATCH, grad_accum=GRAD_ACCUM,
                 effective_batch=BATCH * GRAD_ACCUM, resolution=RES, seed=SEED, split_seed=SPLIT_SEED,
                 train_minutes=round((time.time() - t0) / 60, 2),
                 gpu_hours=round((time.time() - t0) / 3600, 4),
                 peak_mem_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                 final_loss=round(float(np.mean(losses[-200:])), 4), gpu=torch.cuda.get_device_name(0))
    save_adapter(unet, cfg, f"{OUT}/{name}", stats)
    del unet, opt; torch.cuda.empty_cache()
    return stats

TRAIN_STATS = {}
for name in RUN_NOW:
    TRAIN_STATS[name] = train(name, RUNS[name])
    print(name, TRAIN_STATS[name])
for name in RUNS:                                    # pick up a run finished in an earlier session
    p = f"{OUT}/{name}/adapter_config.json"
    if name not in TRAIN_STATS and not os.path.exists(p) and HF_USER:
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(f"{HF_USER}/data266-lab2-p2-ritika-{name.replace('_', '-')}", local_dir=f"{OUT}/{name}",
                              token=HF_TOKEN)
        except Exception as e:
            print(f"{name}: not trained in this session and not on Hugging Face ({type(e).__name__})")
    if os.path.exists(p) and "train_minutes" not in json.load(open(p)):
        os.remove(p)                                 # a mid-training checkpoint, not a finished run
    if name not in TRAIN_STATS and os.path.exists(p):
        TRAIN_STATS[name] = {k: v for k, v in json.load(open(p)).items() if k not in ("rank", "use_dora", "target_modules", "base_model")}
print(pd.DataFrame(TRAIN_STATS).T.to_string())

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

lora_r8 step 100/1200 loss(avg last 100) 0.0547 2.1 min
lora_r8 step 200/1200 loss(avg last 100) 0.0549 4.2 min
lora_r8 step 300/1200 loss(avg last 100) 0.0546 6.4 min
lora_r8 step 400/1200 loss(avg last 100) 0.0496 8.5 min
lora_r8 step 500/1200 loss(avg last 100) 0.0538 10.6 min
lora_r8 step 600/1200 loss(avg last 100) 0.0537 12.7 min
lora_r8 step 700/1200 loss(avg last 100) 0.0527 14.8 min
lora_r8 step 800/1200 loss(avg last 100) 0.0510 16.9 min
lora_r8 step 900/1200 loss(avg last 100) 0.0540 19.0 min
lora_r8 step 1000/1200 loss(avg last 100) 0.0562 21.1 min
lora_r8 step 1100/1200 loss(avg last 100) 0.0535 23.2 min
lora_r8 step 1200/1200 loss(avg last 100) 0.0528 25.3 min
lora_r8 {'trainable_params': 1594368, 'lr': 0.0001, 'steps': 1200, 'batch': 2, 'grad_accum': 2, 'effective_batch': 4, 'resolution': 512, 'seed': 266, 'split_seed': 266, 'train_minutes': 25.34, 'gpu_hours': 0.4223, 'peak_mem_gb': 2.55, 'final_loss': 0.0528, 'gpu': 'Tesla T4'}
dora_r8 step 100/1200 loss(avg last 100) 

In [7]:
# Cell 6: generation. 20 fixed prompts x seeds 0 and 1 for the base model and both runs (40 images each).
from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
from transformers import CLIPImageProcessor

def make_pipe(name=None):
    pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None,
                                                   requires_safety_checker=False).to(DEV)
    if name:
        a = json.load(open(f"{OUT}/{name}/adapter_config.json"))
        pipe.unet.add_adapter(lora_config(a))
        set_peft_model_state_dict(pipe.unet, load_file(f"{OUT}/{name}/adapter_model.safetensors"))
    pipe.set_progress_bar_config(disable=True)
    return pipe

SAFETY = StableDiffusionSafetyChecker.from_pretrained("CompVis/stable-diffusion-safety-checker").to(DEV)
SAFETY_PROC = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

def nsfw_flags(images):
    inp = SAFETY_PROC(images, return_tensors="pt").pixel_values.to(DEV)
    _, flags = SAFETY(images=[np.asarray(i) for i in images], clip_input=inp)
    return flags

MODELS = ["base"] + [n for n in RUNS if os.path.exists(f"{OUT}/{n}/adapter_model.safetensors")]
GEN, GEN_TIME, NSFW = {}, {}, {}
for m in MODELS:
    pipe = make_pipe(None if m == "base" else m)
    os.makedirs(f"{OUT}/images/{m}", exist_ok=True)
    GEN[m], t0, flagged = {}, time.time(), 0
    for p in PROMPTS:
        for s in (0, 1):
            im = pipe(p["prompt"], num_inference_steps=GEN_STEPS, guidance_scale=GUIDANCE,
                      generator=torch.Generator(DEV).manual_seed(s)).images[0]
            if nsfw_flags([im])[0]:                  # NSFW filter: flagged images are blacked out, never published
                im, flagged = Image.new("RGB", im.size), flagged + 1
            im.save(f"{OUT}/images/{m}/p{p['id']:02d}_s{s}.png")
            GEN[m][(p["id"], s)] = im
    GEN_TIME[m], NSFW[m] = (time.time() - t0) / (2 * len(PROMPTS)), flagged
    print(f"{m}: {len(GEN[m])} images, {GEN_TIME[m]:.2f} s/image, NSFW-flagged {flagged}")
    del pipe; torch.cuda.empty_cache()

# side-by-side grids for 8 prompts (columns: models; seed 0)
GRID_IDS = [0, 2, 3, 9, 10, 13, 15, 17]
for gid in GRID_IDS:
    row = Image.new("RGB", (256 * len(MODELS), 256), "white")
    for j, m in enumerate(MODELS):
        row.paste(GEN[m][(gid, 0)].resize((256, 256)), (256 * j, 0))
    row.save(f"{OUT}/images/grid_p{gid:02d}.png")
print("grid columns:", MODELS, "| prompts:", [PROMPTS[g]["prompt"] for g in GRID_IDS])

config.json:   0%|          | 0.00/4.55k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

... (12 outputs left out)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


dora_r8: 40 images, 6.41 s/image, NSFW-flagged 1
grid columns: ['base', 'lora_r8', 'dora_r8'] | prompts: ['a red fish with a gold crown on its head', 'a drawing of a cat wearing a helmet', 'a cartoon of a blue and black dragon', 'a cartoon of a pikachu flying through the air', 'a photograph of a mountain lake at sunrise', 'an astronaut riding a horse on the moon', 'two pokemon, one red and one blue, standing side by side', 'three small round pokemon in a row, the left one purple']


In [9]:
# Cell 7: evaluation. KID vs the test split (in-domain prompts), CLIP score, LPIPS between seeds.
from torchmetrics.image.kid import KernelInceptionDistance
from transformers import CLIPModel, CLIPProcessor
import lpips

# CLIP features via the projection layers directly: works whether get_*_features returns a tensor or an output object
cm = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
cp = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def clip_image_feats(ims):
    out = []
    for i in range(0, len(ims), 32):
        px = cp(images=ims[i:i + 32], return_tensors="pt")["pixel_values"].to(DEV)
        out.append(F.normalize(cm.visual_projection(cm.vision_model(pixel_values=px).pooler_output), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def clip_text_feats(texts):
    t = cp(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEV)
    o = cm.text_model(input_ids=t["input_ids"], attention_mask=t["attention_mask"])
    return F.normalize(cm.text_projection(o.pooler_output), dim=-1)

def to_uint8(ims):
    return torch.stack([torch.from_numpy(np.asarray(i.resize((299, 299)))).permute(2, 0, 1) for i in ims])

in_dom = [p["id"] for p in PROMPTS if p["type"] == "in-domain"]
real = to_uint8([im for im, _ in TEST])
lp = lpips.LPIPS(net="alex", verbose=False).to(DEV)
to_lp = lambda im: (torch.from_numpy(np.asarray(im.resize((256, 256)), dtype=np.float32) / 127.5 - 1).permute(2, 0, 1)[None].to(DEV))

ROWS = []
for m in MODELS:
    fake = [GEN[m][(i, s)] for i in in_dom for s in (0, 1)]
    kid = KernelInceptionDistance(subset_size=min(len(fake), len(TEST)), subsets=100).to(DEV)
    kid.update(real.to(DEV), real=True); kid.update(to_uint8(fake).to(DEV), real=False)
    kid_mean, kid_std = kid.compute()
    clips = {t: [] for t in ("in-domain", "out-of-domain", "compositional")}
    keys = [(p, s) for p in PROMPTS for s in (0, 1)]
    cos = (clip_image_feats([GEN[m][(p["id"], s)] for p, s in keys]) *
           clip_text_feats([p["prompt"] for p, _ in keys])).sum(-1).clamp(min=0)   # CLIP score = max(cos, 0)
    for (p, _), c in zip(keys, cos.tolist()):
        clips[p["type"]].append(c)
    lpips_vals = [lp(to_lp(GEN[m][(p["id"], 0)]), to_lp(GEN[m][(p["id"], 1)])).item() for p in PROMPTS]
    st = TRAIN_STATS.get(m, {})
    ROWS.append({"model": m, "KID (in-domain) mean": round(kid_mean.item(), 5), "KID std": round(kid_std.item(), 5),
                 "CLIP all": round(np.mean(sum(clips.values(), [])), 4),
                 "CLIP in-domain": round(np.mean(clips["in-domain"]), 4),
                 "CLIP out-of-domain": round(np.mean(clips["out-of-domain"]), 4),
                 "CLIP compositional": round(np.mean(clips["compositional"]), 4),
                 "LPIPS seeds": round(np.mean(lpips_vals), 4), "NSFW flagged": NSFW[m],
                 "s / image": round(GEN_TIME[m], 2), "trainable params": st.get("trainable_params", 0),
                 "GPU-hours": st.get("gpu_hours", 0.0), "peak mem GB": st.get("peak_mem_gb", None)})
RESULTS = pd.DataFrame(ROWS).set_index("model")
RESULTS.to_csv(f"{OUT}/p2_results.csv")
print(RESULTS.T.to_string())


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model                    base       lora_r8       dora_r8
KID (in-domain) mean  0.08639  6.081000e-02  5.169000e-02
KID std               0.00859  6.950000e-03  6.170000e-03
CLIP all              0.32230  3.113000e-01  3.156000e-01
CLIP in-domain        0.32610  3.150000e-01  3.254000e-01
CLIP out-of-domain    0.31110  2.984000e-01  3.072000e-01
CLIP compositional    0.32570  3.168000e-01  3.042000e-01
LPIPS seeds           0.73800  7.438000e-01  7.115000e-01
NSFW flagged          0.00000  2.000000e+00  1.000000e+00
s / image             5.51000  5.720000e+00  6.410000e+00
trainable params      0.00000  1.594368e+06  1.694208e+06
GPU-hours             0.00000  4.223000e-01  5.262000e-01
peak mem GB               NaN  2.550000e+00  2.620000e+00


In [14]:
# Cell 7b: the same metrics with NSFW-blacked images removed from every model, so all models are compared on
# the same prompt/seed pairs (a blacked-out image would otherwise count as a real generation).
black = {k for m in MODELS for k, im in GEN[m].items() if np.asarray(im).max() == 0}
print("blacked out in any model (excluded for all):", sorted(black))
rows = []
for m in MODELS:
    fake = [GEN[m][(i, s)] for i in in_dom for s in (0, 1) if (i, s) not in black]
    kid = KernelInceptionDistance(subset_size=min(len(fake), len(TEST)), subsets=100).to(DEV)
    kid.update(real.to(DEV), real=True); kid.update(to_uint8(fake).to(DEV), real=False)
    km, ks = kid.compute()
    keys = [(p, s) for p in PROMPTS for s in (0, 1) if (p["id"], s) not in black]
    cos = (clip_image_feats([GEN[m][(p["id"], s)] for p, s in keys]) *
           clip_text_feats([p["prompt"] for p, _ in keys])).sum(-1).clamp(min=0)
    by_type = {}
    for (p, _), c in zip(keys, cos.tolist()):
        by_type.setdefault(p["type"], []).append(c)
    pairs = [p for p in PROMPTS if (p["id"], 0) not in black and (p["id"], 1) not in black]
    lv = [lp(to_lp(GEN[m][(p["id"], 0)]), to_lp(GEN[m][(p["id"], 1)])).item() for p in pairs]
    rows.append({"model": m, "KID (in-domain) mean": round(km.item(), 5), "KID std": round(ks.item(), 5),
                 "CLIP all": round(float(cos.mean()), 4),
                 **{f"CLIP {t}": round(float(np.mean(v)), 4) for t, v in by_type.items()},
                 "LPIPS seeds": round(float(np.mean(lv)), 4), "in-domain images": len(fake), "images": len(keys)})
CLEAN = pd.DataFrame(rows).set_index("model")
CLEAN.to_csv(f"{OUT}/p2_results_flagged_removed.csv")
print(CLEAN.T.to_string())


blacked out in any model (excluded for all): [(3, 0), (6, 1), (17, 0)]


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


model                     base   lora_r8   dora_r8
KID (in-domain) mean   0.08577   0.05606   0.05352
KID std                0.01124   0.00671   0.00673
CLIP all               0.32150   0.31470   0.31670
CLIP in-domain         0.32310   0.32190   0.32380
CLIP out-of-domain     0.31110   0.29840   0.30720
CLIP compositional     0.32990   0.31830   0.31320
LPIPS seeds            0.73750   0.71100   0.70330
in-domain images      18.00000  18.00000  18.00000
images                37.00000  37.00000  37.00000


In [10]:
# Cell 8: memorization check. Nearest training image (CLIP image embedding cosine) for each in-domain generation.
img_emb = clip_image_feats
train_e = img_emb([im for im, _ in TRAIN])
for m in MODELS[1:]:
    keys = [(i, s) for i in in_dom for s in (0, 1)]
    sims = img_emb([GEN[m][k] for k in keys]) @ train_e.T
    best, arg = sims.max(dim=1)
    order = best.argsort(descending=True)[:3].tolist()
    print(f"{m}: max nearest-train cosine {best.max():.3f}, mean {best.mean():.3f}, "
          f"generations above 0.95: {(best > 0.95).sum().item()} of {len(keys)}")
    pair = Image.new("RGB", (512, 256 * len(order)), "white")
    for r, j in enumerate(order):
        pair.paste(GEN[m][keys[j]].resize((256, 256)), (0, 256 * r))
        pair.paste(TRAIN[arg[j].item()][0].resize((256, 256)), (256, 256 * r))
        print(f"   prompt {keys[j]} cosine {best[j]:.3f} nearest train caption: {TRAIN[arg[j].item()][1]!r}")
    pair.save(f"{OUT}/memorization_{m}.png")


lora_r8: max nearest-train cosine 0.900, mean 0.776, generations above 0.95: 0 of 20
   prompt (9, 1) cosine 0.900 nearest train caption: 'a cartoon pikachu with big eyes and big ears'
   prompt (9, 0) cosine 0.888 nearest train caption: 'a cartoon pikachu with a big smile on his face'
   prompt (8, 1) cosine 0.859 nearest train caption: 'a colorful butterfly that is on a white background'
dora_r8: max nearest-train cosine 0.912, mean 0.798, generations above 0.95: 0 of 20
   prompt (9, 0) cosine 0.912 nearest train caption: 'a cartoon pikachu with a big smile on his face'
   prompt (3, 0) cosine 0.909 nearest train caption: 'a drawing of a blue and yellow pokemon'
   prompt (9, 1) cosine 0.862 nearest train caption: 'a cartoon pikachu with a big smile on his face'


In [15]:
# Cell 9: blinded human evaluation (25 pairs, final run vs base, 3+ raters). This cell writes the pairs;
# after rating, save eval/ritika_neema/p2_human_ratings.csv and re-run the next cell.
# Final run chosen from Cell 7b: KID is tied within one std (LoRA 0.05606 vs DoRA 0.05352, std about 0.0067),
# CLIP differences are small and point both ways, and LoRA r8 trains about 25% faster and generates faster.
FINAL = os.environ.get("P2_FINAL", "lora_r8")
print("final run:", FINAL)
rng = np.random.default_rng(SEED)
cands = [(p["id"], s) for p in PROMPTS for s in (0, 1)]
picked = [cands[i] for i in rng.choice(len(cands), 25, replace=False)]
os.makedirs(f"{OUT}/human_eval", exist_ok=True)
key = []
for n, (pid, s) in enumerate(picked, 1):
    left_is_final = bool(rng.integers(2))
    L, R = (GEN[FINAL], GEN["base"]) if left_is_final else (GEN["base"], GEN[FINAL])
    im = Image.new("RGB", (1024, 512), "white"); im.paste(L[(pid, s)], (0, 0)); im.paste(R[(pid, s)], (512, 0))
    im.save(f"{OUT}/human_eval/pair_{n:02d}.png")
    key.append(dict(pair=n, prompt=PROMPTS[pid]["prompt"], left="final" if left_is_final else "base"))
pd.DataFrame(key).to_csv(f"{OUT}/human_eval/key_hidden.csv", index=False)
pd.DataFrame({"pair": range(1, 26), "prompt": [k["prompt"] for k in key], "rater1": "", "rater2": "",
              "rater3": ""}).to_csv(f"{OUT}/human_eval/ratings_template.csv", index=False)
print("raters answer L, R or T (tie) for the image that better matches the prompt in Pokemon style")

final run: lora_r8
raters answer L, R or T (tie) for the image that better matches the prompt in Pokemon style


In [12]:
# Cell 10: win rate of the final run over the base model from the blinded ratings.
RATINGS = "eval/ritika_neema/p2_human_ratings.csv"
if os.path.exists(RATINGS):
    r = pd.read_csv(RATINGS).merge(pd.read_csv(f"{OUT}/human_eval/key_hidden.csv")[["pair", "left"]], on="pair")
    raters = [c for c in r.columns if c.startswith("rater")]
    votes = []
    for _, row in r.iterrows():
        for c in raters:
            v = str(row[c]).strip().upper()
            votes.append("tie" if v == "T" else "final" if (v == "L") == (row["left"] == "final") else "base")
    v = pd.Series(votes).value_counts()
    print(f"{len(raters)} raters x {len(r)} pairs | final wins {v.get('final', 0)}, base wins {v.get('base', 0)}, "
          f"ties {v.get('tie', 0)} | win rate (ties as half) {(v.get('final', 0) + 0.5 * v.get('tie', 0)) / len(votes):.3f}")
else:
    print("no ratings yet:", RATINGS)

no ratings yet: eval/ritika_neema/p2_human_ratings.csv


In [13]:
# Cell 11: publish. Model cards + adapters to Hugging Face, all images to one dataset repo.
def model_card(name):
    a, row = json.load(open(f"{OUT}/{name}/adapter_config.json")), RESULTS.loc[name]
    kind = "DoRA" if a["use_dora"] else "LoRA"
    return f"""---
license: creativeml-openrail-m
base_model: {BASE}
datasets: [{DATASET}]
tags: [stable-diffusion, text-to-image, {kind.lower()}, data266]
---
# {kind} rank {a['rank']} on Pokemon BLIP captions (DATA 266 Lab 2, Team 05, Ritika Neema)

{kind} adapter (rank {a['rank']}, alpha {a['rank']}) on the UNet attention projections {', '.join(TARGETS)} of
`{BASE}`. Trained on `{DATASET}` (CC BY-NC-SA 4.0: non-commercial use only).

| Setting | Value |
| --- | --- |
| Trainable parameters | {a['trainable_params']:,} |
| Learning rate / optimizer | {a['lr']} / AdamW |
| Steps / batch | {a['steps']} steps, batch {a['batch']} x {a['grad_accum']} accumulation = {a['effective_batch']} |
| Resolution / training seed / split seed | {a['resolution']} / {a['seed']} / {a.get('split_seed', 266)} |
| Peak GPU memory | {a['peak_mem_gb']} GB on {a['gpu']} |
| Training time | {a['train_minutes']} min ({a['gpu_hours']} GPU-hours) |
| KID vs test split (in-domain prompts) | {row['KID (in-domain) mean']} +/- {row['KID std']} |
| CLIP score (all 20 prompts x 2 seeds) | {row['CLIP all']} |
| LPIPS between seeds | {row['LPIPS seeds']} |

Load: create the adapter with `peft.LoraConfig(r={a['rank']}, lora_alpha={a['rank']}, target_modules={TARGETS},
use_dora={a['use_dora']})`, `pipe.unet.add_adapter(config)`, then `set_peft_model_state_dict(pipe.unet, ...)` with
`adapter_model.safetensors`.

Limitations: biased toward the Pokemon art style, weak on out-of-domain prompts. Outputs were screened with the
Stable Diffusion safety checker. Not for commercial use (dataset license).
"""

if API:
    for name in MODELS[1:]:
        open(f"{OUT}/{name}/README.md", "w").write(model_card(name))
        push(f"{OUT}/{name}", f"{HF_USER}/data266-lab2-p2-ritika-{name.replace('_', '-')}", "final adapter and model card")
    img_repo = f"{HF_USER}/data266-lab2-p2-ritika-images"
    API.create_repo(img_repo, repo_type="dataset", exist_ok=True)
    API.upload_folder(folder_path=f"{OUT}/images", repo_id=img_repo, repo_type="dataset", path_in_repo="images",
                      commit_message="120 generated images and comparison grids")
    print("published:", [f"https://huggingface.co/{HF_USER}/data266-lab2-p2-ritika-{n.replace('_', '-')}" for n in MODELS[1:]],
          f"https://huggingface.co/datasets/{img_repo}")
else:
    print("HF_TOKEN not set: nothing published")

published: ['https://huggingface.co/neema30/data266-lab2-p2-ritika-lora-r8', 'https://huggingface.co/neema30/data266-lab2-p2-ritika-dora-r8'] https://huggingface.co/datasets/neema30/data266-lab2-p2-ritika-images


In [16]:

!cd "$OUT/human_eval" && zip -q ../p2_pairs.zip pair_*.png ratings_template.csv
from google.colab import files; files.download(f"{OUT}/p2_pairs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>